# PART 1: Text Processing and Exploratory Data Analysis 

## Task 1. Data preparation

In [7]:
import json, re
import pandas as pd #to see the data in a table format
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

for pkg in ["stopwords", "punkt", "punkt_tab"]:
    nltk.download(pkg, quiet=True)

STOP_WORDS = set(stopwords.words("english"))
STEMMER = PorterStemmer()

def load_docs(path):
    with open(path, encoding="utf-8") as f:
        first = f.read(1); f.seek(0)
        return json.load(f) if first == "[" else [json.loads(l) for l in f if l.strip()]

def load_docs(path):
    with open(path, encoding="utf-8") as f:
        first = f.read(1); f.seek(0)
        return json.load(f) if first == "[" else [json.loads(l) for l in f if l.strip()]

docs = load_docs("fashion_products_dataset.json") 
print(len(docs), "documents")
docs[0]    


28080 documents


{'_id': 'fa8e22d6-c0b6-5229-bb9e-ad52eda39a0a',
 'actual_price': '2,999',
 'average_rating': '3.9',
 'brand': 'York',
 'category': 'Clothing and Accessories',
 'crawled_at': 1612987911000,
 'description': 'Yorker trackpants made from 100% rich combed cotton giving it a rich look.Designed for Comfort,Skin friendly fabric,itch-free waistband & great for all year round use Proudly made in India',
 'discount': '69% off',
 'images': ['https://rukminim1.flixcart.com/image/128/128/jr3t5e80/track-pant/z/y/n/m-1005combo2-yorker-original-imafczg3xfh5qqd4.jpeg?q=70',
  'https://rukminim1.flixcart.com/image/128/128/jr58l8w0/track-pant/w/d/a/l-1005combo8-yorker-original-imafczg3pgtxgraq.jpeg?q=70'],
 'out_of_stock': False,
 'pid': 'TKPFCZ9EA7H5FYZH',
 'product_details': [{'Style Code': '1005COMBO2'},
  {'Closure': 'Elastic'},
  {'Pockets': 'Side Pockets'},
  {'Fabric': 'Cotton Blend'},
  {'Pattern': 'Solid'},
  {'Color': 'Multicolor'}],
 'seller': 'Shyam Enterprises',
 'selling_price': '921',
 'sub

1.1 As a first step, you must pre-process the documents. In particular, for the text fields (title,
description) you should:

    - Removing stop words
    - Tokenization
    - Removing punctuation marks
    - Stemming
    - and... anything else you think it's needed (bonus point)

In [ ]:
# Remove stopwords
def remove_stopwords(tokens):
    return [t for t in tokens if t not in STOP_WORDS]


def tokenize(text):
    return word_tokenize(text)   # or text.split() once punctuation is removed

def remove_punctuation(text):
    # replace with a space so "look.Designed" becomes two tokens
    return re.sub(r"[^a-z0-9\s]", " ", text)

def stem(tokens):
    return [STEMMER.stem(t) for t in tokens]

def normalize(text):
    text = text.lower()                        # case folding
    text = re.sub(r"https?://\S+", " ", text) # remove URLs
    return text

def drop_short(tokens):
    return [t for t in tokens if len(t) > 1 or t.isdigit()]  # drop stray single letters

# Final function calling all of the above
def build_terms(text):
    if not isinstance(text, str) or not text.strip():
        return []
    text = remove_punctuation(normalize(text))
    tokens = text.split()
    tokens = remove_stopwords(tokens)
    tokens = drop_short(tokens)
    return stem(tokens)

#see changes in first document
text = docs[0]["description"]
print(text)
print(build_terms(text))

Yorker trackpants made from 100% rich combed cotton giving it a rich look.Designed for Comfort,Skin friendly fabric,itch-free waistband & great for all year round use Proudly made in India
['yorker', 'trackpant', 'made', '100', 'rich', 'comb', 'cotton', 'give', 'rich', 'look', 'design', 'comfort', 'skin', 'friendli', 'fabric', 'itch', 'free', 'waistband', 'great', 'year', 'round', 'use', 'proudli', 'made', 'india']


1.2 Take into account that for future queries, the final output must return (when present) the
following information for each of the selected documents:
pid, title, description, brand, category, sub_category, product_details, seller, out_of_stock,
selling_price, discount, actual_price, average_rating, url


In [ ]:
OUTPUT_FIELDS = ["pid", "title", "description", "brand", "category", "sub_category",
                 "product_details", "seller", "out_of_stock", "selling_price",
                 "discount", "actual_price", "average_rating", "url"]

def keep_output_fields(doc):
    return {f: doc.get(f) for f in OUTPUT_FIELDS}   # missing fields become None

1.3 Decide how to handle the fields category, sub_category, brand, product_details, and seller
during pre-processing. Should they be merged into a single text field, indexed as separate fields
in the inverted index or any other alternative? Justify your choice, considering how their
distinctiveness may affect retrieval effectiveness. What are pros and cons of each approach?

In [9]:
def flatten_details(details):
    flat = {}
    for item in details or []:
        if isinstance(item, dict):
            flat.update({str(k): str(v) for k, v in item.items()})
    return flat

def details_text(flat):
    skip = {"style code", "sku", "model number", "pack of"}
    return " ".join(f"{k} {v}" for k, v in flat.items() if k.lower() not in skip)

def build_field_terms(doc):
    flat = flatten_details(doc.get("product_details"))
    fields = {
        "title_terms":       build_terms(doc.get("title")),
        "description_terms": build_terms(doc.get("description")),
        "brand_terms":       build_terms(doc.get("brand")),
        "category_terms":    build_terms(f"{doc.get('category') or ''} {doc.get('sub_category') or ''}"),
        "details_terms":     build_terms(details_text(flat)),
        "seller_terms":      build_terms(doc.get("seller")),
    }
    # optional combined field (the "merged" alternative, for comparison in the report)
    fields["all_terms"] = (fields["title_terms"] + fields["description_terms"] + fields["brand_terms"]
                           + fields["category_terms"] + fields["details_terms"])
    return fields

fields = build_field_terms(docs[0])
for name, terms in fields.items():
    print(name, terms)

title_terms ['solid', 'women', 'multicolor', 'track', 'pant']
description_terms ['yorker', 'trackpant', 'made', '100', 'rich', 'comb', 'cotton', 'give', 'rich', 'look', 'design', 'comfort', 'skin', 'friendli', 'fabric', 'itch', 'free', 'waistband', 'great', 'year', 'round', 'use', 'proudli', 'made', 'india']
brand_terms ['york']
category_terms ['cloth', 'accessori', 'bottomwear']
details_terms ['closur', 'elast', 'pocket', 'side', 'pocket', 'fabric', 'cotton', 'blend', 'pattern', 'solid', 'color', 'multicolor']
seller_terms ['shyam', 'enterpris']
all_terms ['solid', 'women', 'multicolor', 'track', 'pant', 'yorker', 'trackpant', 'made', '100', 'rich', 'comb', 'cotton', 'give', 'rich', 'look', 'design', 'comfort', 'skin', 'friendli', 'fabric', 'itch', 'free', 'waistband', 'great', 'year', 'round', 'use', 'proudli', 'made', 'india', 'york', 'cloth', 'accessori', 'bottomwear', 'closur', 'elast', 'pocket', 'side', 'pocket', 'fabric', 'cotton', 'blend', 'pattern', 'solid', 'color', 'multicol

1.4 Consider the fields out_of_stock, selling_price, discount, actual_price, and average_rating.
Decide how these should be handled during pre-processing to use in further search. Should
they be indexed as textual terms?


In [11]:
def to_float(value):
    """'2,999' -> 2999.0 ; '69% off' -> 69.0 ; '' / None -> None"""
    if value is None:
        return None
    m = re.search(r"\d[\d,]*\.?\d*", str(value))
    return float(m.group().replace(",", "")) if m else None

def process_numeric(doc):
    return {
        "out_of_stock":   doc.get("out_of_stock"),            # keep as boolean
        "selling_price":  to_float(doc.get("selling_price")),
        "actual_price":   to_float(doc.get("actual_price")),
        "discount":       to_float(doc.get("discount")),
        "average_rating": to_float(doc.get("average_rating")),
    }

In [12]:
def process_doc(doc):
    out = keep_output_fields(doc)
    out.update(process_numeric(doc))   # overwrites with typed values
    out.update(build_field_terms(doc))
    return out

if __name__ == "__main__":
    docs = load_docs("fashion_products_dataset.json")
    seen, processed = set(), []
    for d in docs:
        if d.get("pid") in seen:
            continue
        seen.add(d.get("pid"))
        processed.append(process_doc(d))
    with open("processed_docs.json", "w", encoding="utf-8") as f:
        json.dump(processed, f, ensure_ascii=False)